# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides an end-to-end guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) Mass Spectrometry dataset using the `mlcroissant` library, referencing all dataset entities by their `@id` fields for robust and FAIR reproducible research.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This ensures all dataset semantics and relationships are respected in the exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print basic metadata
print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, their `@id`s, fields, and field `@id`s, using only the schema's unique identifiers.

In [ ]:
# By design, Croissant datasets expose record sets via the .record_sets attribute.

record_sets = dataset.metadata.record_sets
print(f"Record set @ids and names found in the dataset:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs['name']}")

# For each record set, print its fields and their @ids
for rs in record_sets:
    print(f"\nFields in record set {rs['name']} (@id: {rs['@id']}):")
    for field in rs['fields']:
        fname = field['name'] if 'name' in field else field.get('@id','<unnamed>')
        print(f"  - {fname} (@id: {field['@id']})")

## 3. Data Extraction
Load data from **all** record sets into DataFrames for analysis. Use `@id` for record sets and fields, as enumerated above.

*Note: You may need to consult the overview output above to select meaningful IDs for your workflow.*

In [ ]:
# Prepare a dictionary of DataFrames for all record sets
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]
print(f"Processing these record set @ids: {record_set_ids}")

for record_set_id in record_set_ids:
    # Load all records from this record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Display column names for the first record set
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"First record set @id: {first_rs}")
    print(f"Columns available (@id): {dataframes[first_rs].columns.tolist()}")
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing such as filtering, normalization, or grouping using selected field `@id`s from the previous section. **Always reference columns by the field `@id`, not by display name.**

In [ ]:
# Example: Pick a record set and numeric field by @id
# Replace these with actual string @ids shown by the Overview section above

# (For demonstration): Choose the first record set with non-empty DataFrame
chosen_record_set_id = None
for rid, df in dataframes.items():
    if not df.empty:
        chosen_record_set_id = rid
        break

if chosen_record_set_id is not None:
    chosen_df = dataframes[chosen_record_set_id]
    print(f"Using record set: {chosen_record_set_id}")

    # Try to identify a numeric field by inferring dtype
    sample_numeric_fields = [c for c in chosen_df.columns if pd.api.types.is_numeric_dtype(chosen_df[c])]
    if not sample_numeric_fields:
        print("No numeric field found in DataFrame. Please check data overview for a correct field @id.")
    else:
        numeric_field_id = sample_numeric_fields[0]
        print(f"Numeric field selected (@id): {numeric_field_id}")

        # Filtering
        threshold = chosen_df[numeric_field_id].mean() if pd.notnull(chosen_df[numeric_field_id].mean()) else 10
        filtered_df = chosen_df[chosen_df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.3f} (threshold = mean):")
        display(filtered_df.head())

        # Normalization
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, normalized_col]].head())

        # Group by the first available object-type column that isn't the numeric one
        group_candidates = [c for c in filtered_df.columns if c != numeric_field_id and pd.api.types.is_object_dtype(filtered_df[c])]
        if group_candidates:
            group_field_id = group_candidates[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
            display(grouped_df.head())
        else:
            print("No suitable categorical field to group by.")
else:
    print("No record set with data found.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalization, as well as the means by group if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and not filtered_df.empty:
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    sns.histplot(filtered_df[numeric_field_id], bins=20, kde=True)
    plt.title(f"{numeric_field_id} distribution (filtered)")

    plt.subplot(1, 2, 2)
    sns.histplot(filtered_df[normalized_col], bins=20, kde=True, color='orange')
    plt.title(f"{numeric_field_id} normalized distribution (filtered)")
    plt.show()

    # If a group_field exists, barplot
    if 'grouped_df' in locals():
        plt.figure(figsize=(8,4))
        sns.barplot(data=grouped_df, x=grouped_df.columns[0], y=grouped_df.columns[1], color='skyblue')
        plt.xticks(rotation=45)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()
else:
    print("No filtered_df available for visualization.")

## 6. Conclusion
In this notebook, we have:
- Loaded the FAIR^2 Mass Spectrometry dataset using its Croissant schema and the `mlcroissant` library
- Explored its structure by `@id`, listing all record sets and fields
- Extracted tabular data and performed basic filtering, normalization, and grouping using only field and record set `@id`s
- Visualized field distributions and group means

Next steps could include more domain-specific analysis, leveraging the dataset for protein abundance and modification studies.

> ⚠️ Make sure to consult the dataset and schema documentation to interpret each field's `@id` semantics properly in further research.